## Classification

To run this script, you need the following files found in the /data directory:
- "final_labels_SG1.xlsx"
- "final_labels_SG2.xlsx"

## Imports and set-up

In [ ]:
!pip install transformers
!pip install sentencepiece
!pip install datasets
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 18.1 MB/s eta 0:00:00

In [ ]:
# Resolve paths relative to the repository root so this notebook runs from the repo
# root, from notebooks/, or on Colab after cloning the repo:
#   !git clone https://github.com/yasmeenhany/media-bias-detection.git
#   %cd media-bias-detection
import os
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
os.chdir(REPO_ROOT)
print("Working directory:", REPO_ROOT)

In [ ]:

!nvidia-smi

Sun Dec  8 13:07:47 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   45C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
import sys
import os
import time
import re
import random
from typing import Dict, List, Optional, Union
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import tensorflow as tf
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
tf.random.set_seed(0)
random.seed(0)
np.random.seed(0)
os.environ['PYTHONHASHSEED']=str(0)
os.environ['TF_DETERMINISTIC_OPS'] = '0'

In [ ]:
# see if hardware accelerator available
tf.config.experimental.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
tf.test.gpu_device_name()

'/device:GPU:0'

If GPUs are available, tensorflow will give priority to it automatically and computations will be performed on the GPU as default. That behavior can be changed by assigning a task explicitly to a device. Example:

```
with tf.device('/CPU:0'):
```



## Preprocessing

In [ ]:
PATH_sg1 = "data/final_labels_SG1.xlsx"
PATH_sg2 = "data/final_labels_SG2.xlsx"
df_sg1 = pd.read_excel(PATH_sg1)
df_sg2 = pd.read_excel(PATH_sg2)
df_sg1.rename(columns={'text': 'sentence', 'label_bias': 'Label_bias'}, inplace=True)
df_sg2.rename(columns={'text': 'sentence', 'label_bias': 'Label_bias'}, inplace=True)
df_sg1.head()

,sentence,news_link,outlet,topic,type,Label_bias,label_opinion,biased_words
0,The Republican president assumed he was helpin...,http://www.msnbc.com/rachel-maddow-show/auto-i...,msnbc,environment,left,Biased,Expresses writer’s opinion,[]
1,Though the indictment of a woman for her own p...,https://eu.usatoday.com/story/news/nation/2019...,usa-today,abortion,center,Non-biased,Somewhat factual but also opinionated,[]
2,Ingraham began the exchange by noting American...,https://www.breitbart.com/economy/2020/01/12/d...,breitbart,immigration,right,No agreement,No agreement,['flood']
3,The tragedy of America’s 18 years in Afghanist...,http://feedproxy.google.com/~r/breitbart/~3/ER...,breitbart,international-politics-and-world-news,right,Biased,Somewhat factual but also opinionated,"['tragedy', 'stubborn']"
4,The justices threw out a challenge from gun ri...,https://www.huffpost.com/entry/supreme-court-g...,msnbc,gun-control,left,Non-biased,Entirely factual,[]


In [ ]:

# binarize classification problem
df_sg1 = df_sg1[df_sg1['Label_bias']!='No agreement']
df_sg1 = df_sg1[df_sg1['Label_bias'].isna()==False]
df_sg1.replace(to_replace='Biased', value=1, inplace=True)
df_sg1.replace(to_replace='Non-biased', value=0, inplace=True)

df_sg2 = df_sg2[df_sg2['Label_bias']!='No agreement']
df_sg2.replace(to_replace='Biased', value=1, inplace=True)
df_sg2.replace(to_replace='Non-biased', value=0, inplace=True)

<ipython-input-11-3610be668419>:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sg1.replace(to_replace='Non-biased', value=0, inplace=True)
<ipython-input-11-3610be668419>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sg2.replace(to_replace='Biased', value=1, inplace=True)
<ipython-input-11-3610be668419>:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

The rest of the preprocessing needs to be performed inside the folds as a) encoder layers shouldn't be allowed to see whole data to construct the lookups and b) indexing with skfold is not possible when data is in tensorflow format.

In [ ]:
# helper functions called in skfold loop

def pd_to_tf(df):
    """convert a pandas dataframe into a tensorflow dataset"""
    target = df.pop('Label_bias')
    sentence = df.pop('sentence')
    return tf.data.Dataset.from_tensor_slices((sentence.values, target.values))

def plot_graphs(history, metric):
  plt.plot(history.history[metric])
  plt.plot(history.history['val_'+metric], '')
  plt.xlabel("Epochs")
  plt.ylabel(metric)
  plt.legend([metric, 'val_'+metric])
  plt.show()

def tokenize(df):
    """convert a pandas dataframe into a tensorflow dataset and run hugging face's tokenizer on data"""
    target = df.pop('Label_bias')
    sentence = df.pop('sentence')

    train_encodings = tokenizer(
                        sentence.tolist(),
                        add_special_tokens = True, # add [CLS], [SEP]
                        truncation = True, # cut off at max length of the text that can go to BERT
                        padding = True, # add [PAD] tokens
                        return_attention_mask = True, # add attention mask to not focus on pad tokens
              )

    dataset = tf.data.Dataset.from_tensor_slices(
        (dict(train_encodings),
         target.tolist()))
    return dataset

## Attention-based models


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
import torch

def run_model_5fold(df_train, model_name="microsoft/deberta-v3-small", freeze_encoder=False, pretrained=False, plot=False, tuning=False, n_splits=5):

    Y = df_train['Label_bias']
    X = df_train['sentence']
    skfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    k = 1

    val_loss = []
    val_acc = []
    val_prec = []
    val_rec = []
    val_f1 = []
    val_f1_micro = []
    val_f1_wmacro = []

    for train_index, val_index in skfold.split(X, Y):
        print(f'### Start fold {k}')

        train_dataset = df_train.iloc[train_index]
        val_dataset = df_train.iloc[val_index]

        train_dataset = train_dataset[['sentence', 'Label_bias']]
        val_dataset = val_dataset[['sentence', 'Label_bias']]

        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

        def tokenize_data(data):
            return tokenizer(
                data['sentence'].tolist(),
                padding=True,
                truncation=True,
                max_length=150,
                return_tensors="pt"
            )

        train_encodings = tokenize_data(train_dataset)
        val_encodings = tokenize_data(val_dataset)

        train_labels = torch.tensor(train_dataset['Label_bias'].tolist())
        val_labels = torch.tensor(val_dataset['Label_bias'].tolist())

        train_data = Dataset.from_dict({
            'input_ids': train_encodings['input_ids'].numpy(),
            'attention_mask': train_encodings['attention_mask'].numpy(),
            'labels': train_labels.numpy()
        })

        val_data = Dataset.from_dict({
            'input_ids': val_encodings['input_ids'].numpy(),
            'attention_mask': val_encodings['attention_mask'].numpy(),
            'labels': val_labels.numpy()
        })

        def model_init():
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=len(Y.unique())
            )
            if freeze_encoder:
                for param in model.base_model.parameters():
                    param.requires_grad = False
            return model

        training_args = TrainingArguments(
            output_dir=f'./results/fold_{k}',
            evaluation_strategy="epoch",
            learning_rate=5e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=3,
            weight_decay=0.1,
            seed=42
        )

        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=1)
            precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
            f1_micro = precision_recall_fscore_support(labels, preds, average='micro')[2]
            f1_macro = precision_recall_fscore_support(labels, preds, average='macro')[2]

            acc = accuracy_score(labels, preds)
            return {
                'accuracy': acc,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'f1_micro': f1_micro,
                'f1_macro': f1_macro
            }

        if tuning:
          def hyperparameter_search_space(trial):
              return {
                  "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
                  "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
                  "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 5),
                  "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
              }

          trainer = Trainer(
                model_init=model_init,
                args=training_args,
                train_dataset=train_data,
                eval_dataset=val_data,
                tokenizer=tokenizer,
                compute_metrics=compute_metrics
            )

          best_run = trainer.hyperparameter_search(
                direction="maximize",
                n_trials=45,
                compute_objective=lambda metrics: metrics["eval_f1"],
                hp_space=hyperparameter_search_space
            )

          print(f"Best Hyperparameters for Fold {k}: {best_run.hyperparameters}")
          for key, value in best_run.hyperparameters.items():
              setattr(training_args, key, value)

        else:
            trainer = Trainer(
                model_init=model_init,
                args=training_args,
                train_dataset=train_data,
                eval_dataset=val_data,
                tokenizer=tokenizer,
                compute_metrics=compute_metrics
            )

        trainer.train()

        eval_result = trainer.evaluate()
        val_loss.append(eval_result.get('eval_loss', None))
        val_acc.append(eval_result.get('eval_accuracy', None))
        val_prec.append(eval_result.get('eval_precision', None))
        val_rec.append(eval_result.get('eval_recall', None))
        val_f1.append(eval_result.get('eval_f1', None))
        val_f1_micro.append(eval_result.get('eval_f1_micro', None))
        val_f1_wmacro.append(eval_result.get('eval_f1_macro', None))

        k += 1

    return val_loss, val_acc, val_prec, val_rec, val_f1, val_f1_micro, val_f1_wmacro


\deBERTa

In [ ]:

import os
os.environ["WANDB_DISABLED"] = "true"

val_loss, val_acc, val_prec, val_rec, val_f1, val_f1_micro, val_f1_wmacro = run_model_5fold(df_sg1,freeze_encoder=False, pretrained=False, tuning=False)

loss_cv = np.mean(val_loss)
acc_cv = np.mean(val_acc)
prec_cv = np.mean(val_prec)
rec_cv = np.mean(val_rec)
f1_cv = np.mean(val_f1)

f1_micro_cv = np.mean(val_f1_micro)
f1_wmacro_cv = np.mean(val_f1_wmacro)

print('5-Fold CV Loss: {}'.format(loss_cv))
print('5-Fold CV Accuracy: {}'.format(acc_cv))
print('5-Fold CV Precision: {}'.format(prec_cv))
print('5-Fold CV Recall: {}'.format(rec_cv))
print('5-Fold CV F1 Score: {}'.format(f1_cv))
print('5-Fold CV Micro F1 Score: {}'.format(f1_micro_cv))
print('5-Fold CV Weighted Macro F1 Score: {}'.format(f1_wmacro_cv))

### Start fold 1


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-16-c63c9e118a95>:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Micro,F1 Macro
1,No log,0.480656,0.770968,0.771456,0.770968,0.770559,0.770968,0.770104
2,No log,0.396126,0.845161,0.850400,0.845161,0.844185,0.845161,0.843697
3,No log,0.427861,0.841935,0.842799,0.841935,0.841653,0.841935,0.841339


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Micro,F1 Macro
1,No log,0.588622,0.731392,0.751274,0.731392,0.727898,0.731392,0.728836
2,No log,0.623343,0.737864,0.748702,0.737864,0.733344,0.737864,0.731923
3,No log,0.646522,0.741100,0.741023,0.741100,0.740937,0.741100,0.740489


### Start fold 5


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-16-c63c9e118a95>:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSeq

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Micro,F1 Macro
1,No log,0.414813,0.822006,0.822383,0.822006,0.821771,0.822006,0.821400
2,No log,0.410734,0.831715,0.836950,0.831715,0.831493,0.831715,0.831629
3,No log,0.502000,0.828479,0.833876,0.828479,0.827259,0.828479,0.826620


5-Fold CV Loss: 0.5376749515533448
5-Fold CV Accuracy: 0.8065747990395657
5-Fold CV Precision: 0.8085479857670184
5-Fold CV Recall: 0.8065747990395657
5-Fold CV F1 Score: 0.8059944726541479
5-Fold CV Micro F1 Score: 0.8065747990395657
5-Fold CV Weighted Macro F1 Score: 0.8055324896978704


In [ ]:
val_loss, val_acc, val_prec, val_rec, val_f1, val_f1_micro, val_f1_wmacro = run_model_5fold(df_sg2,freeze_encoder=False, pretrained=False, tuning=False)

loss_cv = np.mean(val_loss)
acc_cv = np.mean(val_acc)
prec_cv = np.mean(val_prec)
rec_cv = np.mean(val_rec)
f1_cv = np.mean(val_f1)

f1_micro_cv = np.mean(val_f1_micro)
f1_wmacro_cv = np.mean(val_f1_wmacro)

print('5-Fold CV Loss: {}'.format(loss_cv))
print('5-Fold CV Accuracy: {}'.format(acc_cv))
print('5-Fold CV Precision: {}'.format(prec_cv))
print('5-Fold CV Recall: {}'.format(rec_cv))
print('5-Fold CV F1 Score: {}'.format(f1_cv))
print('5-Fold CV Micro F1 Score: {}'.format(f1_micro_cv))
print('5-Fold CV Weighted Macro F1 Score: {}'.format(f1_wmacro_cv))

### Start fold 1


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-20-c63c9e118a95>:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSeq

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Micro,F1 Macro
1,No log,0.491075,0.771429,0.782262,0.771429,0.769670,0.771429,0.769946
2,No log,0.458813,0.808163,0.814314,0.808163,0.806950,0.808163,0.806698
3,0.392200,0.537913,0.802721,0.803929,0.802721,0.802392,0.802721,0.802247


### Start fold 2


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-20-c63c9e118a95>:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSeq

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Micro,F1 Macro
1,No log,0.484000,0.784741,0.788848,0.784741,0.784203,0.784741,0.784331
2,No log,0.540702,0.787466,0.787775,0.787466,0.787333,0.787466,0.787238
3,0.382600,0.635167,0.792916,0.792920,0.792916,0.792894,0.792916,0.792840


### Start fold 5


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-20-c63c9e118a95>:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSeq

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Micro,F1 Macro
1,No log,0.442337,0.792916,0.792948,0.792916,0.792879,0.792916,0.792817
2,No log,0.485895,0.799728,0.799889,0.799728,0.799735,0.799728,0.799724
3,0.371800,0.586732,0.806540,0.808136,0.806540,0.806156,0.806540,0.806020


5-Fold CV Loss: 0.5653020024299622
5-Fold CV Accuracy: 0.8121359061335707
5-Fold CV Precision: 0.8130554573222701
5-Fold CV Recall: 0.8121359061335707
5-Fold CV F1 Score: 0.8119294218056791
5-Fold CV Micro F1 Score: 0.8121359061335707
5-Fold CV Weighted Macro F1 Score: 0.8118335206927805
